<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

folder_path = '/content/drive/MyDrive/fraud_detection_project'

# Check if the folder exists
if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' was not found.")
    print("Please ensure the folder exists in your Google Drive and the path is correct.")
    print("Here are the contents of your 'MyDrive' folder:")
    try:
        my_drive_contents = os.listdir('/content/drive/MyDrive')
        for item in my_drive_contents:
            print(f" - {item}")
    except Exception as e:
        print(f"Could not list MyDrive contents: {e}")
else:
    files = os.listdir(folder_path)

    print("Files found in your project folder:")
    if files:
        for f in files:
            print(" -", f)
    else:
        print(" - No files found in this folder.")

Files found in your project folder:
 - test_identity.csv
 - train_identity.csv
 - test_transaction.csv
 - train_transaction.csv
 - cleaned_data.csv
 - X_train.npy
 - X_test.npy
 - y_train.npy
 - y_test.npy
 - scaler.pkl
 - feature_columns.pkl
 - baseline_confusion_matrix.png
 - baseline_model.pkl
 - baseline_results.txt
 - federated_learning_results.png
 - federated_model.pkl
 - global_weights.pkl
 - federated_results.txt
 - encrypted_coef.pkl
 - encrypted_intercept.pkl
 - he_context.pkl
 - model_comparison.png
 - privacy_vs_performance.png
 - final_results.txt
 - app.py


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pickle

In [ ]:
print("Loading train_transaction.csv... please wait")
train_transaction = pd.read_csv('/content/drive/MyDrive/fraud_detection_project/train_transaction.csv')
print("Done! Shape:", train_transaction.shape)

print("Loading train_identity.csv... please wait")
train_identity = pd.read_csv('/content/drive/MyDrive/fraud_detection_project/train_identity.csv')
print("Done! Shape:", train_identity.shape)

Loading train_transaction.csv... please wait
Done! Shape: (590540, 394)
Loading train_identity.csv... please wait
Done! Shape: (144233, 41)


In [ ]:
# Merging on TransactionID — like joining two excel sheets
df = pd.merge(train_transaction, train_identity,
              on='TransactionID',
              how='left')

print("Merged shape:", df.shape)
print("Total transactions:", len(df))
print("Fraud cases:", df['isFraud'].sum())
print("Non-fraud cases:", (df['isFraud'] == 0).sum())
print("Fraud percentage:", round(df['isFraud'].mean() * 100, 2), "%")

Merged shape: (590540, 434)
Total transactions: 590540
Fraud cases: 20663
Non-fraud cases: 569877
Fraud percentage: 3.5 %


In [ ]:
# Check how many missing values each column has
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

# Show only columns that have missing values
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percent': missing_percent
})

missing_df = missing_df[missing_df['Missing Count'] > 0]
missing_df = missing_df.sort_values('Missing Percent', ascending=False)

print("Total columns with missing values:", len(missing_df))
print("\nTop 20 columns with most missing values:")
print(missing_df.head(20))

Total columns with missing values: 414

Top 20 columns with most missing values:
       Missing Count  Missing Percent
id_24         585793        99.196159
id_25         585408        99.130965
id_07         585385        99.127070
id_08         585385        99.127070
id_21         585381        99.126393
id_26         585377        99.125715
id_23         585371        99.124699
id_22         585371        99.124699
id_27         585371        99.124699
dist2         552913        93.628374
D7            551623        93.409930
id_18         545427        92.360721
D13           528588        89.509263
D14           528353        89.469469
D12           525823        89.041047
id_04         524216        88.768923
id_03         524216        88.768923
D6            517353        87.606767
id_33         517251        87.589494
id_10         515614        87.312290


In [ ]:
# Drop columns with more than 70% missing values
threshold = 70
cols_to_drop = missing_df[missing_df['Missing Percent'] > threshold].index.tolist()


protected_cols = ['TransactionID', 'isFraud']
cols_to_drop = [col for col in cols_to_drop if col not in protected_cols]

df_cleaned = df.drop(columns=cols_to_drop)

print("Shape before dropping:", df.shape)
print("Shape after dropping:", df_cleaned.shape)
print("Columns remaining:", df_cleaned.shape[1])

Shape before dropping: (590540, 434)
Shape after dropping: (590540, 226)
Columns remaining: 226


In [ ]:
# For remaining missing values
# Fill numerical columns with median
# Fill categorical columns with mode (most common value)

for col in df_cleaned.columns:
    if df_cleaned[col].dtype == 'object':
        mode_value = df_cleaned[col].mode()[0]
        df_cleaned[col] = df_cleaned[col].fillna(mode_value)
    else:
        median_value = df_cleaned[col].median()
        df_cleaned[col] = df_cleaned[col].fillna(median_value)

# Verify no missing values remain
remaining_missing = df_cleaned.isnull().sum().sum()
print("Remaining missing values:", remaining_missing)
print("Shape after filling:", df_cleaned.shape)

Remaining missing values: 0
Shape after filling: (590540, 226)


In [ ]:
# Convert text columns to numbers
# ML models only understand numbers, not text like "desktop" or "visa"

categorical_cols = df_cleaned.select_dtypes(include=['object']).columns
print("Categorical columns found:", len(categorical_cols))

for col in categorical_cols:
    le = LabelEncoder()
    df_cleaned[col] = le.fit_transform(df_cleaned[col].astype(str))

print("\nAll categorical columns converted to numbers!")
print("Shape:", df_cleaned.shape)

Categorical columns found: 13

All categorical columns converted to numbers!
Shape: (590540, 226)


In [ ]:
# X = input features (everything except isFraud)
# y = what we want to predict (isFraud)

X = df_cleaned.drop(columns=['isFraud', 'TransactionID'])
y = df_cleaned['isFraud']

print("Features shape (X):", X.shape)
print("Target shape (y):", y.shape)
print("Fraud cases in target:", y.sum())

Features shape (X): (590540, 224)
Target shape (y): (590540,)
Fraud cases in target: 20663


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for testing
    random_state=42,      # same split every time
    stratify=y            # keep same fraud ratio in both splits
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("Fraud in training:", y_train.sum())
print("Fraud in testing:", y_test.sum())

Training set size: (472432, 224)
Testing set size: (118108, 224)
Fraud in training: 16530
Fraud in testing: 4133


In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
import numpy as np
import pickle

# ============================================
# STEP 1: Split into train and test
# ============================================

X = df_cleaned.drop(columns=['isFraud', 'TransactionID'])
y = df_cleaned['isFraud']

print("Features shape (X):", X.shape)
print("Target shape (y):", y.shape)
print("Fraud cases in target:", y.sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain-test split done!")
print("Training set size:", X_train.shape)
print("Testing set size :", X_test.shape)
print("Fraud in training:", y_train.sum())
print("Fraud in testing :", y_test.sum())

# ============================================
# STEP 2: Check for non-numeric columns
# ============================================

non_numeric_cols = X_train.select_dtypes(include=['object']).columns.tolist()

if len(non_numeric_cols) > 0:
    print("\n⚠️ Non-numeric columns still found:")
    print(non_numeric_cols)
else:
    print("\n✅ All columns are numeric. Safe for SMOTE.")

# ============================================
# STEP 3: Apply CONTROLLED SMOTE ONLY on training data
# ============================================

from imblearn.over_sampling import SMOTE

target_fraud_samples = 100000   # safer for RAM

smote = SMOTE(
    sampling_strategy={1: target_fraud_samples},
    random_state=42
)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nBefore SMOTE:")
print("Original training shape:", X_train.shape)
print("Original fraud count   :", int(y_train.sum()))
print("Original non-fraud     :", int((y_train == 0).sum()))

print("\nAfter CONTROLLED SMOTE:")
print("Resampled training shape:", X_train_resampled.shape)
print("Resampled fraud count   :", int(y_train_resampled.sum()))
print("Resampled non-fraud     :", int((y_train_resampled == 0).sum()))

# ============================================
# STEP 4: Scale the data
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled  = scaler.transform(X_test)

print("\nScaling done!")
print("Training data scaled shape:", X_train_scaled.shape)
print("Testing data scaled shape :", X_test_scaled.shape)

# ============================================
# STEP 5: Save processed files
# ============================================

df_cleaned.to_csv('/content/drive/MyDrive/fraud_detection_project/cleaned_data.csv', index=False)
print("\ncleaned_data.csv saved!")

np.save('/content/drive/MyDrive/fraud_detection_project/X_train.npy', X_train_scaled)
np.save('/content/drive/MyDrive/fraud_detection_project/X_test.npy', X_test_scaled)
np.save('/content/drive/MyDrive/fraud_detection_project/y_train.npy', np.array(y_train_resampled))
np.save('/content/drive/MyDrive/fraud_detection_project/y_test.npy', np.array(y_test))

with open('/content/drive/MyDrive/fraud_detection_project/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('/content/drive/MyDrive/fraud_detection_project/feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)

print("X_train.npy saved!")
print("X_test.npy saved!")
print("y_train.npy saved!")
print("y_test.npy saved!")
print("scaler.pkl saved!")
print("feature_columns.pkl saved!")

print("\n✅ Preprocessing with CONTROLLED SMOTE completed successfully!")

Features shape (X): (590540, 224)
Target shape (y): (590540,)
Fraud cases in target: 20663

Train-test split done!
Training set size: (472432, 224)
Testing set size : (118108, 224)
Fraud in training: 16530
Fraud in testing : 4133

✅ All columns are numeric. Safe for SMOTE.

Before SMOTE:
Original training shape: (472432, 224)
Original fraud count   : 16530
Original non-fraud     : 455902

After CONTROLLED SMOTE:
Resampled training shape: (555902, 224)
Resampled fraud count   : 100000
Resampled non-fraud     : 455902

Scaling done!
Training data scaled shape: (555902, 224)
Testing data scaled shape : (118108, 224)

cleaned_data.csv saved!
X_train.npy saved!
X_test.npy saved!
y_train.npy saved!
y_test.npy saved!
scaler.pkl saved!
feature_columns.pkl saved!

✅ Preprocessing with CONTROLLED SMOTE completed successfully!


In [ ]:
import pickle
import numpy as np

# Save cleaned dataframe
df_cleaned.to_csv('/content/drive/MyDrive/fraud_detection_project/cleaned_data.csv', index=False)
print("cleaned_data.csv saved!")

# Save scaled arrays as numpy files
np.save('/content/drive/MyDrive/fraud_detection_project/X_train.npy', X_train_scaled)
np.save('/content/drive/MyDrive/fraud_detection_project/X_test.npy', X_test_scaled)
np.save('/content/drive/MyDrive/fraud_detection_project/y_train.npy', np.array(y_train_resampled))
np.save('/content/drive/MyDrive/fraud_detection_project/y_test.npy', np.array(y_test))

# Save scaler for later use
with open('/content/drive/MyDrive/fraud_detection_project/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save column names for later use
with open('/content/drive/MyDrive/fraud_detection_project/feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)

print("X_train.npy saved!")
print("X_test.npy saved!")
print("y_train.npy saved!")
print("y_test.npy saved!")
print("scaler.pkl saved!")
print("feature_columns.pkl saved!")

print("\nAll preprocessing files saved to Drive!")

cleaned_data.csv saved!
X_train.npy saved!
X_test.npy saved!
y_train.npy saved!
y_test.npy saved!
scaler.pkl saved!
feature_columns.pkl saved!

All preprocessing files saved to Drive!
